In [1]:
import random
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# ============================
# Helper: Calculate Charge & GRAVY
# ============================
kd_scale = {
    'A': 1.8,
    'R': -4.5,
    'N': -3.5,
    'D': -3.5,
    'C': 2.5,
    'Q': -3.5,
    'E': -3.5,
    'G': -0.4,
    'H': -3.2,
    'I': 4.5,
    'L': 3.8,
    'K': -3.9,
    'M': 1.9,
    'F': 2.8,
    'P': -1.6,
    'S': -0.8,
    'T': -0.7,
    'W': -0.9,
    'Y': -1.3,
    'V': 4.2,
}


def get_physicochemical_features(seq):
  # Net charge at pH 7.0 (K, R = +1; D, E = -1)
  charge = (
      seq.count('K') + seq.count('R') - seq.count('D') - seq.count('E')
  )
  # GRAVY score
  valid_aas = [aa for aa in seq if aa in kd_scale]
  gravy = (
      sum(kd_scale[aa] for aa in valid_aas) / len(valid_aas) if valid_aas else 0
  )
  return [charge, gravy]


# ============================
# Baseline & Dataset Export Loop
# ============================
positive_files = [
    'pos_1 filtered_gravy.xlsx',
    'pos_9 filtered_charge.xlsx',
    'pos_filtered_charge_gravy.xlsx',
]
negative_files = ['neg1_charge_gravy.xlsx', 'neg2_charge_gravy.xlsx']

baseline_results = []

print('\n--- Starting Baseline Evaluation & Train/Test File Generation ---')

for pi, pos_file in enumerate(positive_files, start=1):
  for ni, neg_file in enumerate(negative_files, start=1):

    # Set exact base seed to reproduce master training script split
    base_seed = 1000 + pi * 10 + ni
    np.random.seed(base_seed)
    random.seed(base_seed)

    # Load master files
    df_pos = pd.read_excel(pos_file)
    df_neg = pd.read_excel(neg_file)

    df_pos['label'] = 1
    df_neg['label'] = 0

    df_all = pd.concat(
        [df_pos[['Sequence', 'label']], df_neg[['Sequence', 'label']]],
        ignore_index=True,
    )

    # Sequence strings and labels for dataset export
    X_seqs = df_all['Sequence'].values
    y = df_all['label'].values

    # Compute explicit 2-feature scalar array for baseline models
    X_feats = np.array([
        get_physicochemical_features(s) for s in df_all['Sequence']
    ])

    # Perform reproducible train-test split on both sequences and features
    (
        X_train_seq,
        X_test_seq,
        X_train_feat,
        X_test_feat,
        y_train,
        y_test,
    ) = train_test_split(
        X_seqs,
        X_feats,
        y,
        test_size=0.2,
        stratify=y,
        random_state=base_seed,
        shuffle=True,
    )

    # ============================================================
    # 1. EXPORT TRAIN AND TEST CSV FILES FOR AUDITING
    # ============================================================
    df_train_export = pd.DataFrame({'Sequence': X_train_seq, 'Label': y_train})
    df_test_export = pd.DataFrame({'Sequence': X_test_seq, 'Label': y_test})

    train_filename = f'dataset_P{pi}_N{ni}_TRAIN.csv'
    test_filename = f'dataset_P{pi}_N{ni}_TEST.csv'

    df_train_export.to_csv(train_filename, index=False)
    df_test_export.to_csv(test_filename, index=False)

    print(
        f'[P{pi}+N{ni}] Exported: {train_filename} ({len(df_train_export)}'
        f' seqs) | {test_filename} ({len(df_test_export)} seqs)'
    )

    # ============================================================
    # 2. RUN BASELINE MODELS (LOGISTIC REGRESSION & RANDOM FOREST)
    # ============================================================
    # Logistic Regression Baseline
    lr = LogisticRegression()
    lr.fit(X_train_feat, y_train)
    lr_probs = lr.predict_proba(X_test_feat)[:, 1]
    lr_auc = roc_auc_score(y_test, lr_probs)

    # Random Forest Baseline
    rf = RandomForestClassifier(n_estimators=100, random_state=base_seed)
    rf.fit(X_train_feat, y_train)
    rf_probs = rf.predict_proba(X_test_feat)[:, 1]
    rf_auc = roc_auc_score(y_test, rf_probs)

    baseline_results.append({
        'Classifier': f'P{pi}+N{ni}',
        'Train Size': len(X_train_seq),
        'Test Size': len(X_test_seq),
        'Logistic Regression AUC (Charge+GRAVY Only)': round(lr_auc, 4),
        'Random Forest AUC (Charge+GRAVY Only)': round(rf_auc, 4),
    })

# Save final baseline comparison summary
df_base_summary = pd.DataFrame(baseline_results)
df_base_summary.to_csv('baseline_comparison_results.csv', index=False)

print('\n--- All Train/Test files and Baseline AUC results saved! ---')
print(df_base_summary)


--- Starting Baseline Evaluation & Train/Test File Generation ---
[P1+N1] Exported: dataset_P1_N1_TRAIN.csv (640 seqs) | dataset_P1_N1_TEST.csv (161 seqs)
[P1+N2] Exported: dataset_P1_N2_TRAIN.csv (437 seqs) | dataset_P1_N2_TEST.csv (110 seqs)
[P2+N1] Exported: dataset_P2_N1_TRAIN.csv (746 seqs) | dataset_P2_N1_TEST.csv (187 seqs)
[P2+N2] Exported: dataset_P2_N2_TRAIN.csv (543 seqs) | dataset_P2_N2_TEST.csv (136 seqs)
[P3+N1] Exported: dataset_P3_N1_TRAIN.csv (609 seqs) | dataset_P3_N1_TEST.csv (153 seqs)
[P3+N2] Exported: dataset_P3_N2_TRAIN.csv (406 seqs) | dataset_P3_N2_TEST.csv (102 seqs)

--- All Train/Test files and Baseline AUC results saved! ---
  Classifier  Train Size  Test Size  \
0      P1+N1         640        161   
1      P1+N2         437        110   
2      P2+N1         746        187   
3      P2+N2         543        136   
4      P3+N1         609        153   
5      P3+N2         406        102   

   Logistic Regression AUC (Charge+GRAVY Only)  \
0            

In [3]:
"""
Ensemble Random Forest Baseline Evaluation on generated_peptides_20.csv
======================================================================
This script:
1. Loops through all 6 classifier pairs (P1+N1 to P3+N2).
2. Computes Charge and GRAVY for training and generated peptides.
3. Trains 6 Random Forest models on the respective split datasets.
4. Generates prediction probabilities across all 6 models for generated_peptides_20.csv.
5. Computes the Ensemble Mean RF Probability and statistical summary.
6. Exports results to Excel and generates a distribution plot.
"""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# --- 1. Kyte-Doolittle Hydropathy Scale ---
KD_SCALE = {
    'A': 1.8,
    'R': -4.5,
    'N': -3.5,
    'D': -3.5,
    'C': 2.5,
    'Q': -3.5,
    'E': -3.5,
    'G': -0.4,
    'H': -3.2,
    'I': 4.5,
    'L': 3.8,
    'K': -3.9,
    'M': 1.9,
    'F': 2.8,
    'P': -1.6,
    'S': -0.8,
    'T': -0.7,
    'W': -0.9,
    'Y': -1.3,
    'V': 4.2,
}


def extract_physicochemical_features(sequence):
  """Computes Net Charge and GRAVY for a given amino acid sequence."""
  seq = str(sequence).upper().strip()
  charge = seq.count('K') + seq.count('R') - seq.count('D') - seq.count('E')
  valid_aas = [aa for aa in seq if aa in KD_SCALE]
  gravy = (
      sum(KD_SCALE[aa] for aa in valid_aas) / len(valid_aas) if valid_aas else 0.0
  )
  return [charge, gravy]


# --- 2. Load Generated Peptides ---
generated_file = 'generated_peptides_20.csv'
print(f'=== Loading Generated Peptides ({generated_file}) ===')
df_gen = pd.read_csv(generated_file)

# Detect sequence column
possible_cols = ['Sequence', 'sequence', 'seq', 'Peptide']
seq_col = next((col for col in possible_cols if col in df_gen.columns), None)
if seq_col is None:
  seq_col = df_gen.columns[0]

print(f'Loaded {len(df_gen)} generated sequences. Using column: "{seq_col}"')

# Extract features for generated peptides
X_gen = np.array(
    [extract_physicochemical_features(s) for s in df_gen[seq_col]]
)
df_gen['Net_Charge'] = X_gen[:, 0]
df_gen['GRAVY'] = X_gen[:, 1]

# --- 3. Loop Across All 6 Classifier Pairs ---
rf_prob_columns = []

print('\n=== Running Random Forest Baseline across All 6 Model Pairs ===')

for p_idx in range(1, 4):
  for n_idx in range(1, 3):
    pair_name = f'P{p_idx}_N{n_idx}'
    train_file = f'dataset_{pair_name}_TRAIN.csv'
    test_file = f'dataset_{pair_name}_TEST.csv'

    # Load respective split files
    df_train = pd.read_csv(train_file)
    df_test = pd.read_csv(test_file)

    # Prepare features
    X_train = np.array([
        extract_physicochemical_features(s) for s in df_train['Sequence']
    ])
    y_train = df_train['Label'].values

    X_test = np.array([
        extract_physicochemical_features(s) for s in df_test['Sequence']
    ])
    y_test = df_test['Label'].values

    # Train Random Forest
    base_seed = 1000 + p_idx * 10 + n_idx
    rf = RandomForestClassifier(
        n_estimators=100, max_depth=5, random_state=base_seed
    )
    rf.fit(X_train, y_train)

    # Evaluate hold-out test AUC
    test_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

    # Predict on generated candidate library
    col_name = f'RF_Prob_{pair_name}'
    df_gen[col_name] = rf.predict_proba(X_gen)[:, 1]
    rf_prob_columns.append(col_name)

    print(
        f'[{pair_name}] Hold-out Test AUC: {test_auc:.4f} | Predicted on'
        ' candidates.'
    )

# --- 4. Calculate Ensemble Mean Score ---
df_gen['RF_Ensemble_Mean_Prob'] = df_gen[rf_prob_columns].mean(axis=1)

# --- 5. Statistical Summary ---
print('\n======================================================')
print('=== RF Ensemble Prediction Summary on Candidates ===')
print('======================================================')
stats = df_gen['RF_Ensemble_Mean_Prob'].describe()
print(stats)

# Save distribution plot
plt.figure(figsize=(7, 4), dpi=300)
sns.histplot(
    df_gen['RF_Ensemble_Mean_Prob'],
    bins=30,
    kde=True,
    color='darkred',
    edgecolor='black',
)
plt.title(f'RF Ensemble Mean Score Distribution (N={len(df_gen)})')
plt.xlabel('Ensemble Mean Activity Probability (Random Forest)')
plt.ylabel('Candidate Peptide Count')
plt.tight_layout()
plt.savefig('RF_Ensemble_Generated_Peptides_Distribution.png')
plt.close()

# Export final Excel file
output_file = 'generated_peptides_20_RF_Ensemble_assessed.xlsx'
df_gen.to_excel(output_file, index=False)
print(f'\nFull results saved to: {output_file}')
print("Distribution plot saved as: 'RF_Ensemble_Generated_Peptides_Distribution.png'")

=== Loading Generated Peptides (generated_peptides_20.csv) ===
Loaded 3000 generated sequences. Using column: "Sequence"

=== Running Random Forest Baseline across All 6 Model Pairs ===
[P1_N1] Hold-out Test AUC: 1.0000 | Predicted on candidates.
[P1_N2] Hold-out Test AUC: 1.0000 | Predicted on candidates.
[P2_N1] Hold-out Test AUC: 1.0000 | Predicted on candidates.
[P2_N2] Hold-out Test AUC: 0.9942 | Predicted on candidates.
[P3_N1] Hold-out Test AUC: 1.0000 | Predicted on candidates.
[P3_N2] Hold-out Test AUC: 1.0000 | Predicted on candidates.

=== RF Ensemble Prediction Summary on Candidates ===
count    3000.000000
mean        0.599980
std         0.367177
min         0.001757
25%         0.180490
50%         0.656608
75%         1.000000
max         1.000000
Name: RF_Ensemble_Mean_Prob, dtype: float64

Full results saved to: generated_peptides_20_RF_Ensemble_assessed.xlsx
Distribution plot saved as: 'RF_Ensemble_Generated_Peptides_Distribution.png'


In [4]:
"""
Comprehensive Sequence Novelty and Diversity Pipeline for Reviewer Comment 6
===========================================================================
This script:
1. Loads all positive training sequences across P1, P2, and P3 training splits.
2. Combines them into a master deduplicated positive reference database.
3. Loads all generated peptides (from generated_peptides_20.csv).
4. Computes Nearest-Neighbor sequence identity using Biopython global sequence alignment.
5. Exports a FASTA file of generated peptides formatted for CD-HIT clustering.
6. Prints summary statistics ready for Table S3.
"""

import os
from Bio import Align
import numpy as np
import pandas as pd

# --- Step 1: Combine All Positive Training Datasets (P1, P2, P3) ---
train_files = [
    "dataset_P1_N1_TRAIN.csv",
    "dataset_P2_N1_TRAIN.csv",
    "dataset_P3_N1_TRAIN.csv",
]
pos_seqs_set = set()

print("=== Step 1: Building Combined Positive Reference Database ===")
for tf in train_files:
  if os.path.exists(tf):
    df_t = pd.read_csv(tf)
    # Filter for positive class (Label == 1)
    pos = (
        df_t[df_t["Label"] == 1]["Sequence"]
        .dropna()
        .astype(str)
        .str.upper()
        .str.strip()
        .tolist()
    )
    pos_seqs_set.update(pos)
    print(f"Loaded {len(pos)} positive sequences from {tf}.")
  else:
    print(f"Notice: File '{tf}' not found in local directory. Skipping...")

master_train_pos = list(pos_seqs_set)
print(
    "\nTotal Unique Positive Reference Sequences across ALL Models:"
    f" {len(master_train_pos)}"
)

# --- Step 2: Load Generated Peptides ---
gen_file = "generated_peptides_20.csv"
print(f"\n=== Step 2: Loading Generated Candidates ({gen_file}) ===")
df_gen = pd.read_csv(gen_file)

possible_cols = ["Sequence", "sequence", "seq", "Peptide"]
seq_col = next((col for col in possible_cols if col in df_gen.columns), None)
if seq_col is None:
  seq_col = df_gen.columns[0]

gen_seqs = (
    df_gen[seq_col]
    .dropna()
    .astype(str)
    .str.upper()
    .str.strip()
    .unique()
    .tolist()
)
print(f"Loaded {len(gen_seqs)} unique generated peptide sequences.")

# --- Step 3: Biopython Global Alignment Setup ---
aligner = Align.PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 1.0
aligner.mismatch_score = 0.0
aligner.open_gap_score = -1.0
aligner.extend_gap_score = -0.5


def calculate_sequence_identity(seq1, seq2):
  """Calculates percentage identity based on max length global alignment."""
  score = aligner.score(seq1, seq2)
  max_len = max(len(seq1), len(seq2))
  return (score / max_len) * 100.0


def get_nearest_neighbor_identity(query_seq, target_database):
  """Finds maximum sequence identity against master training database."""
  max_id = 0.0
  for target in target_database:
    # Length difference heuristic to optimize alignment speed
    if abs(len(query_seq) - len(target)) > 10:
      continue
    ident = calculate_sequence_identity(query_seq, target)
    if ident > max_id:
      max_id = ident
      if max_id == 100.0:
        break
  return max_id


# --- Step 4: Compute Nearest-Neighbor Identities ---
print(
    "\n=== Step 3: Computing Nearest-Neighbor Identity across Master"
    " Reference Database ==="
)
nn_identities = []
for i, g_seq in enumerate(gen_seqs):
  if (i + 1) % 500 == 0 or (i + 1) == len(gen_seqs):
    print(f"Processed {i + 1}/{len(gen_seqs)} candidate sequences...")
  nn_id = get_nearest_neighbor_identity(g_seq, master_train_pos)
  nn_identities.append(nn_id)

nn_identities = np.array(nn_identities)

# --- Step 5: Export FASTA File for CD-HIT Clustering ---
fasta_out = "generated_peptides_for_cdhit.fasta"
with open(fasta_out, "w") as f:
  for idx, seq in enumerate(gen_seqs):
    f.write(f">Candidate_{idx+1}\n{seq}\n")

print(
    f"\nFASTA file exported for CD-HIT sequence clustering: '{fasta_out}'"
)

# --- Step 6: Print Summary Statistics ---
mean_nn = np.mean(nn_identities)
std_nn = np.std(nn_identities)
max_nn = np.max(nn_identities)
pct_less_80 = (np.sum(nn_identities < 80.0) / len(nn_identities)) * 100.0
pct_exact_100 = (np.sum(nn_identities == 100.0) / len(nn_identities)) * 100.0

print("\n" + "=" * 65)
print("   FINAL SEQUENCE NOVELTY METRICS (ALL POSITIVE TRAINING POOLS)")
print("=" * 65)
print(f"Total Generated Candidates Assessed  : {len(gen_seqs)}")
print(
    "Master Positive Database Size        :"
    f" {len(master_train_pos)} unique sequences"
)
print(f"Mean Nearest-Neighbor Identity       : {mean_nn:.2f}% ± {std_nn:.2f}%")
print(f"Maximum Identity to Training Set     : {max_nn:.2f}%")
print(f"% Candidates with < 80% Identity      : {pct_less_80:.2f}%")
print(f"% Exact Matches (100% Identity)       : {pct_exact_100:.2f}%")
print("=" * 65)

=== Step 1: Building Combined Positive Reference Database ===
Loaded 239 positive sequences from dataset_P1_N1_TRAIN.csv.
Loaded 345 positive sequences from dataset_P2_N1_TRAIN.csv.
Loaded 208 positive sequences from dataset_P3_N1_TRAIN.csv.

Total Unique Positive Reference Sequences across ALL Models: 429

=== Step 2: Loading Generated Candidates (generated_peptides_20.csv) ===
Loaded 3000 unique generated peptide sequences.

=== Step 3: Computing Nearest-Neighbor Identity across Master Reference Database ===
Processed 500/3000 candidate sequences...
Processed 1000/3000 candidate sequences...
Processed 1500/3000 candidate sequences...
Processed 2000/3000 candidate sequences...
Processed 2500/3000 candidate sequences...
Processed 3000/3000 candidate sequences...

FASTA file exported for CD-HIT sequence clustering: 'generated_peptides_for_cdhit.fasta'

   FINAL SEQUENCE NOVELTY METRICS (ALL POSITIVE TRAINING POOLS)
Total Generated Candidates Assessed  : 3000
Master Positive Database Siz

In [6]:
"""
Comprehensive Sequence Novelty and Diversity Pipeline (Generative Seed Source)
===========================================================================
This script:
1. Loads the exact Excel file used for generation (pos_charge_gravy.xlsx).
2. Deduplicates and cleans positive reference sequences.
3. Loads generated peptides (generated_peptides_20.csv).
4. Computes Nearest-Neighbor sequence identity using Biopython global alignment (Needleman-Wunsch).
5. Exports 'generated_peptides_for_cdhit.fasta' for CD-HIT clustering.
6. Prints summary statistics formatted for Table S3.
"""

import os
from Bio import Align
import numpy as np
import pandas as pd

# --- Step 1: Load Generative Seed Dataset (pos_charge_gravy.xlsx) ---
seed_file = "pos_charge_gravy.xlsx"
print(f"=== Step 1: Loading Generative Seed Source Dataset ({seed_file}) ===")

if not os.path.exists(seed_file):
  raise FileNotFoundError(
      f"Error: '{seed_file}' not found in the current directory. Please place"
      " it in the same folder."
  )

# Read Excel file
df_seed = pd.read_excel(seed_file)

# Auto-detect sequence column in pos_charge_gravy.xlsx
possible_cols = [
    "Sequence",
    "sequence",
    "seq",
    "Peptide",
    "peptides",
    "pos_seq",
    "Generated_Sequence",
]
seq_col_seed = next(
    (col for col in possible_cols if col in df_seed.columns), None
)
if seq_col_seed is None:
  seq_col_seed = df_seed.columns[0]  # Fallback to first column

master_train_pos = (
    df_seed[seq_col_seed]
    .dropna()
    .astype(str)
    .str.upper()
    .str.strip()
    .unique()
    .tolist()
)

print(
    f"Loaded {len(master_train_pos)} unique seed sequences from '{seed_file}'"
    f" using column '{seq_col_seed}'."
)

# --- Step 2: Load Generated Peptides ---
gen_file = "generated_peptides_20.csv"
print(f"\n=== Step 2: Loading Generated Candidates ({gen_file}) ===")
df_gen = pd.read_csv(gen_file)

seq_col_gen = next(
    (col for col in possible_cols if col in df_gen.columns), None
)
if seq_col_gen is None:
  seq_col_gen = df_gen.columns[0]

gen_seqs = (
    df_gen[seq_col_gen]
    .dropna()
    .astype(str)
    .str.upper()
    .str.strip()
    .unique()
    .tolist()
)
print(f"Loaded {len(gen_seqs)} unique generated peptide sequences.")

# --- Step 3: Biopython Global Alignment Setup ---
aligner = Align.PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 1.0
aligner.mismatch_score = 0.0
aligner.open_gap_score = -1.0
aligner.extend_gap_score = -0.5


def calculate_sequence_identity(seq1, seq2):
  """Calculates percentage identity based on max length global alignment."""
  score = aligner.score(seq1, seq2)
  max_len = max(len(seq1), len(seq2))
  return (score / max_len) * 100.0


def get_nearest_neighbor_identity(query_seq, target_database):
  """Finds maximum sequence identity against seed database."""
  max_id = 0.0
  for target in target_database:
    # Length difference heuristic to optimize alignment speed
    if abs(len(query_seq) - len(target)) > 10:
      continue
    ident = calculate_sequence_identity(query_seq, target)
    if ident > max_id:
      max_id = ident
      if max_id == 100.0:
        break
  return max_id


# --- Step 4: Compute Nearest-Neighbor Identities ---
print(
    "\n=== Step 3: Computing Nearest-Neighbor Identity against"
    f" {seed_file} ==="
)
nn_identities = []
for i, g_seq in enumerate(gen_seqs):
  if (i + 1) % 500 == 0 or (i + 1) == len(gen_seqs):
    print(f"Processed {i + 1}/{len(gen_seqs)} candidate sequences...")
  nn_id = get_nearest_neighbor_identity(g_seq, master_train_pos)
  nn_identities.append(nn_id)

nn_identities = np.array(nn_identities)

# --- Step 5: Export FASTA File for CD-HIT Clustering ---
fasta_out = "generated_peptides_for_cdhit.fasta"
with open(fasta_out, "w") as f:
  for idx, seq in enumerate(gen_seqs):
    f.write(f">Candidate_{idx+1}\n{seq}\n")

print(
    f"\nFASTA file exported for CD-HIT sequence clustering: '{fasta_out}'"
)

# --- Step 6: Print Summary Statistics ---
mean_nn = np.mean(nn_identities)
std_nn = np.std(nn_identities)
max_nn = np.max(nn_identities)
pct_less_80 = (np.sum(nn_identities < 80.0) / len(nn_identities)) * 100.0
pct_exact_100 = (np.sum(nn_identities == 100.0) / len(nn_identities)) * 100.0

print("\n" + "=" * 65)
print(f"  FINAL SEQUENCE NOVELTY METRICS (SOURCE: {seed_file})")
print("=" * 65)
print(f"Total Generated Candidates Assessed  : {len(gen_seqs)}")
print(
    "Generative Seed Database Size        :"
    f" {len(master_train_pos)} unique sequences"
)
print(f"Mean Nearest-Neighbor Identity       : {mean_nn:.2f}% ± {std_nn:.2f}%")
print(f"Maximum Identity to Seed Set         : {max_nn:.2f}%")
print(f"% Candidates with < 80% Identity      : {pct_less_80:.2f}%")
print(f"% Exact Matches (100% Identity)       : {pct_exact_100:.2f}%")
print("=" * 65)

=== Step 1: Loading Generative Seed Source Dataset (pos_charge_gravy.xlsx) ===
Loaded 632 unique seed sequences from 'pos_charge_gravy.xlsx' using column 'Sequence'.

=== Step 2: Loading Generated Candidates (generated_peptides_20.csv) ===
Loaded 3000 unique generated peptide sequences.

=== Step 3: Computing Nearest-Neighbor Identity against pos_charge_gravy.xlsx ===
Processed 500/3000 candidate sequences...
Processed 1000/3000 candidate sequences...
Processed 1500/3000 candidate sequences...
Processed 2000/3000 candidate sequences...
Processed 2500/3000 candidate sequences...
Processed 3000/3000 candidate sequences...

FASTA file exported for CD-HIT sequence clustering: 'generated_peptides_for_cdhit.fasta'

  FINAL SEQUENCE NOVELTY METRICS (SOURCE: pos_charge_gravy.xlsx)
Total Generated Candidates Assessed  : 3000
Generative Seed Database Size        : 632 unique sequences
Mean Nearest-Neighbor Identity       : 31.54% ± 11.90%
Maximum Identity to Seed Set         : 95.00%
% Candidate

In [8]:
import numpy as np
from Bio import Align

# 1. Load generated FASTA sequences
fasta_file = "generated_peptides_for_cdhit.fasta"
sequences = []
with open(fasta_file, "r") as f:
  current_seq = ""
  for line in f:
    if line.startswith(">"):
      if current_seq:
        sequences.append(current_seq)
        current_seq = ""
    else:
      current_seq += line.strip()
  if current_seq:
    sequences.append(current_seq)

print(f"Loaded {len(sequences)} sequences for clustering.")

# 2. Setup Biopython Aligner
aligner = Align.PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 1.0
aligner.mismatch_score = 0.0
aligner.open_gap_score = -1.0
aligner.extend_gap_score = -0.5


def seq_identity(s1, s2):
  score = aligner.score(s1, s2)
  max_l = max(len(s1), len(s2))
  return (score / max_l) * 100.0


# 3. Greedy Clustering at 80% Identity Threshold
clusters = []  # List of cluster representative sequences
threshold = 80.0

print("Clustering sequences at 80% identity threshold...")
for i, seq in enumerate(sequences):
  if (i + 1) % 500 == 0 or (i + 1) == len(sequences):
    print(f"Clustered {i + 1}/{len(sequences)} sequences...")

  matched = False
  for rep in clusters:
    # Length check filter for speed
    if abs(len(seq) - len(rep)) > 6:
      continue
    if seq_identity(seq, rep) >= threshold:
      matched = True
      break

  if not matched:
    clusters.append(seq)

print("\n" + "=" * 55)
print(f"   PURE PYTHON CD-HIT EQUIVALENT CLUSTER COUNT")
print("=" * 55)
print(f"Total Sequences Evaluated : {len(sequences)}")
print(f"Clustering Threshold      : 80.0%")
print(f"Total Unique Clusters     : {len(clusters)}")
print("=" * 55)

Loaded 3000 sequences for clustering.
Clustering sequences at 80% identity threshold...
Clustered 500/3000 sequences...
Clustered 1000/3000 sequences...
Clustered 1500/3000 sequences...
Clustered 2000/3000 sequences...
Clustered 2500/3000 sequences...
Clustered 3000/3000 sequences...

   PURE PYTHON CD-HIT EQUIVALENT CLUSTER COUNT
Total Sequences Evaluated : 3000
Clustering Threshold      : 80.0%
Total Unique Clusters     : 2847
